# Lesion review — diseased coronary arteries

Finds every narrowing in each diseased coronary tree, checks it against the mesh and the CT, picks the most severe, and draws one review page per case for manual sign-off.

**How to use:** run sections 1–5 once (they only define things). Then run section 6 for a single case, or section 7 for all cases, and section 8 to rebuild the combined tables. Each section can be re-run on its own; after changing a setting in section 2, re-run section 2 and then the case you want.

Outputs go in this notebook's folder: `Diseased_N_review.png`, `Diseased_N_lesions.csv`, `lesions_all.csv`, `picks.csv`, `flag_definitions.csv`. `overrides.csv` is created once and never overwritten.

Needs: numpy, scipy, openpyxl, trimesh, shapely, matplotlib (and pandas for the tables in section 9).

### Method
```
  1. Every centreline path (ostium -> branch tip) gives stations with position and inscribed-sphere
     radius r. r is averaged over 1 mm (the lumen is only a few voxels wide, so single stations are noisy).
  2. Healthy reference = a tapering curve fitted through the path (local linear fit, 10 mm half-width),
     refitted 5 times ignoring stations that are narrowed (< 85% of the fit) or bulging (> 130%).
     It is fitted separately between bifurcations, because a daughter branch is smaller than its parent,
     and the 3 mm either side of each bifurcation (the flare) is left out of the fit. Downstream pieces
     with less than 10 mm of usable vessel are fitted together with the piece upstream of them.
  3. Narrowing = 1 - r_smoothed / reference. A lesion is any stretch narrowed > 25%; lesions less than
     3 mm apart are merged (tandem lesions). Paths share trunks, so duplicates are merged by throat position.
  4. Each lesion is checked against the mesh cross-section area and the CT: calcium volume (voxels
     brighter than max(500 HU, lumen mean + 3 SD)) within the reference radius + 1.5 mm of the lesion.
  5. Branch tips: a branch may end where the annotators stopped (vessel under 1 mm or imaging artefact),
     not where the artery ends. A narrowing only counts if the vessel is seen healthy again (< 15% narrowed
     for >= 1 mm) past it; otherwise it is flagged "tip". Lesions with < 5 mm of vessel past them are
     flagged "runoff" so you can check them.
  6. Pick = most severe lesion that is not at the ostium or a branch tip and whose mesh
     cross-section narrowing agrees with the radius narrowing to within 20 points (otherwise the radius is
     probably an artefact). The reported severity is only compared, never used to choose.
```

## 1. Imports

In [ ]:
import base64, csv, glob, gzip, math, os, re, struct, sys, zlib, shutil, tempfile, time, warnings
import xml.etree.ElementTree as ET
import numpy as np
import openpyxl
import trimesh
from scipy.spatial import cKDTree
from scipy.ndimage import map_coordinates
from shapely.geometry import Point
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

## 2. Paths and settings
`HERE` is the folder this notebook is in (the `LesionReview` folder); `DISEASED` is the folder above it. Change `DISEASED` if you move the notebook.

In [ ]:
HERE = os.getcwd()
DISEASED = os.path.dirname(HERE)
print('Diseased folder:', DISEASED)

SMOOTH_MM = 1.0        # radius averaging window
BANDWIDTH_MM = 10.0    # reference fit half-width
LESION_DS = 0.25       # narrowing that counts as a lesion
MERGE_GAP_MM = 3.0     # merge lesions closer than this
MIN_SEG_MM = 10.0      # shortest branch piece that gets its own reference fit
MIN_LEN_MM = 0.5       # ignore narrowings shorter than this
NEAR_MM = 3.0          # "near" ostium / tip / bifurcation
RECOVER_DS = 0.15      # vessel counts as recovered below this narrowing ...
RECOVER_MM = 1.0      # ... for at least this long past the lesion
SHORT_RUNOFF_MM = 5.0  # flag lesions with less vessel than this past them
SHELL_MM = 1.5         # CT search distance beyond the reference radius
CA_MIN_HU = 500        # calcium threshold floor
CPR_HALF_MM = 10.0     # straightened CT view: +/- this around the throat
MIN_REPORTED = 0.0     # cases reported below this are skipped (0 = all 20 diseased cases)

## 3. File readers
Centreline `.vtp` (VTK XML, zlib-compressed binary), and `.nrrd` CT / annotation volumes.

In [ ]:
def decode(da):
    txt = da.text.strip()
    nblk = struct.unpack("<I", base64.b64decode(txt[:8])[:4])[0]
    hlen = math.ceil((12 + 4 * nblk) / 3) * 4
    sizes = struct.unpack_from(f"<{nblk}I", base64.b64decode(txt[:hlen]), 12)
    body, raw, pos = base64.b64decode(txt[hlen:]), b"", 0
    for s in sizes:
        raw += zlib.decompress(body[pos:pos + s]); pos += s
    dt = {"Float32": "<f4", "Float64": "<f8", "Int64": "<i8", "Int32": "<i4"}[da.attrib["type"]]
    return np.frombuffer(raw, dtype=dt).reshape(-1, int(da.attrib.get("NumberOfComponents", 1))).squeeze()

def load_paths(vtp):
    piece = ET.parse(vtp).getroot().find(".//Piece")
    radius = decode(next(d for d in piece.find("PointData") if d.attrib["Name"] == "MaximumInscribedSphereRadius"))
    pts = decode(piece.find("Points/DataArray")).astype(float)
    conn, offs = (decode(d) for d in piece.find("Lines"))
    out = []
    for a, b in zip(np.r_[0, offs[:-1]], offs):
        idx = conn[a:b]
        z = np.r_[0, np.cumsum(np.linalg.norm(np.diff(pts[idx], axis=0), axis=1))]
        out.append((z, radius[idx].astype(float), pts[idx]))
    return out

def read_nrrd(path):
    raw = open(path, "rb").read()
    i = raw.index(b"\n\n")
    h = {}
    for line in raw[:i].decode(errors="ignore").splitlines()[1:]:
        if line.startswith("#") or ":" not in line: continue
        k, v = line.split(":", 1); h[k.strip()] = v.strip()
    data = gzip.decompress(raw[i + 2:]) if h.get("encoding") == "gzip" else raw[i + 2:]
    dt = {"int16": "<i2", "short": "<i2", "int32": "<i4", "int": "<i4", "uint8": "<u1", "float": "<f4"}[h["type"]]
    a = np.frombuffer(data, dtype=dt).reshape([int(s) for s in h["sizes"].split()][::-1])
    sp = np.array([float(v) for v in re.findall(r"[-\d.e]+", h["space directions"])]).reshape(3, 3).diagonal()
    org = np.array([float(v) for v in re.findall(r"[-\d.e]+", h["space origin"])])
    return a, sp, org

## 4. Profile analysis
Radius averaging, the tapering healthy-reference fit, lesion detection, and mesh cross-sections.

In [ ]:
def moving_avg(z, r, w):
    lo = np.searchsorted(z, z - w / 2, "left"); hi = np.searchsorted(z, z + w / 2, "right")
    c = np.r_[0, np.cumsum(r)]
    return (c[hi] - c[lo]) / (hi - lo)

def reference(z, rs, w0=None):
    w0 = np.ones_like(rs) if w0 is None else w0
    w_rob = w0.copy()
    lo = np.searchsorted(z, z - BANDWIDTH_MM, "left"); hi = np.searchsorted(z, z + BANDWIDTH_MM, "right")
    for _ in range(5):
        fit = np.empty_like(rs)
        for i in range(len(z)):
            zz, rr = z[lo[i]:hi[i]], rs[lo[i]:hi[i]]
            k = np.clip(1 - (np.abs(zz - z[i]) / BANDWIDTH_MM) ** 3, 0, None) ** 3
            w = k * w_rob[lo[i]:hi[i]]
            if w.sum() < 1e-3 * k.sum(): w = k
            sw = w.sum(); zm = (w * zz).sum() / sw; rm = (w * rr).sum() / sw
            var = (w * (zz - zm) ** 2).sum()
            b = (w * (zz - zm) * (rr - rm)).sum() / var if var > 1e-9 else 0.0
            fit[i] = rm + b * (z[i] - zm)
        w_rob = ((rs >= 0.85 * fit) & (rs <= 1.30 * fit)).astype(float) * w0
    return fit

def find_lesions(z, ds):
    on = ds > LESION_DS
    segs, i = [], 0
    while i < len(z):
        if on[i]:
            j = i
            while j + 1 < len(z) and on[j + 1]: j += 1
            segs.append([i, j]); i = j + 1
        else:
            i += 1
    merged = []
    for s in segs:
        if merged and z[s[0]] - z[merged[-1][1]] < MERGE_GAP_MM: merged[-1][1] = s[1]
        else: merged.append(s)
    return [s for s in merged if z[s[1]] - z[s[0]] >= MIN_LEN_MM]

def tangent(p, i, span=1.0):
    d = np.linalg.norm(p - p[i], axis=1)
    lo, hi = max(0, i - 1), min(len(p) - 1, i + 1)
    while lo > 0 and d[lo] < span: lo -= 1
    while hi < len(p) - 1 and d[hi] < span: hi += 1
    t = p[hi] - p[lo]
    return t / np.linalg.norm(t)

def section_eq_r(mesh, o, t):
    s = mesh.section(plane_origin=o, plane_normal=t)
    if s is None: return np.nan
    try:
        pl, T = s.to_planar()
    except Exception:
        return np.nan
    q = Point(trimesh.transform_points([o], np.linalg.inv(T))[0][:2])
    # smallest closed outline around the centreline point = this vessel's lumen
    # (polygons_closed, unlike polygons_full, does not need the optional rtree package)
    hits = [pg for pg in pl.polygons_closed if pg is not None and pg.contains(q)]
    return math.sqrt(min(pg.area for pg in hits) / math.pi) if hits else np.nan

In [ ]:
def path_profiles(name):
    """Every centreline path of a case with its smoothed radius, healthy reference and narrowing,
    plus the bifurcation points. Used by the lesion review and by the cutting step."""
    paths = load_paths(os.path.join(DISEASED, "Centerlines", f"{name}.vtp"))
    # bifurcations: where each path leaves the paths before it
    bifs, trees = [], []
    for z, r, p in paths:
        if trees:
            d = np.min([t.query(p)[0] for t in trees], axis=0)
            shared = d < 0.05
            if shared[0] and not shared.all():
                bifs.append(p[np.argmin(shared) - 1])
        trees.append(cKDTree(p))
    bifs = np.array(bifs) if bifs else np.zeros((0, 3))

    prof = []
    for z, r, p in paths:
        rs = moving_avg(z, r, SMOOTH_MM)
        # a daughter branch is legitimately smaller than its parent, so the reference is fitted
        # separately between bifurcations instead of being smoothed across the size step
        cuts = sorted({int(np.argmin(np.linalg.norm(p - bp, axis=1))) for bp in bifs
                       if np.min(np.linalg.norm(p - bp, axis=1)) < 0.3})
        cuts = [c for c in cuts if 0 < c < len(z) - 1]
        # a downstream piece too short to show its own healthy size (< MIN_SEG_MM once the flares are
        # left out) is fitted together with the piece upstream of it
        while True:
            edges = [0] + cuts + [len(z)]
            w0 = np.ones_like(rs)
            for c in cuts: w0[np.abs(z - z[c]) < NEAR_MM] = 0
            usable = [np.diff(z[a:e])[w0[a:e - 1] > 0].sum() if e - a > 1 else 0 for a, e in zip(edges[:-1], edges[1:])]
            short = [i for i, u in enumerate(usable) if u < MIN_SEG_MM and i > 0]   # the trunk piece may be short
            if not short: break
            cuts.pop(short[0] - 1)          # merge the short piece into the piece upstream of it
        ref = np.empty_like(rs)
        for a, e in zip(edges[:-1], edges[1:]):
            ww = w0[a:e] if w0[a:e].sum() >= 5 else np.ones(e - a)
            seg = reference(z[a:e], rs[a:e], ww) if e - a > 5 else rs[a:e].copy()
            ref[a:e] = np.clip(seg, 0.5 * np.median(rs[a:e]), 1.5 * rs[a:e].max())   # guard against extrapolation
        ds = 1 - rs / ref
        prof.append(dict(z=z, r=r, p=p, rs=rs, ref=ref, ds=ds, cuts=[z[c] for c in cuts]))
    return paths, prof, bifs

## 5. CT helpers, per-case analysis and outputs
Coordinate mapping to CT voxels, calcium check, straightened CT views, `analyse()` for one case, and the functions that write the review page and the tables.

In [ ]:
def to_index(P, sp, org):
    """mesh / centreline coordinates -> voxel index (x, y, z). x and y carry a flipped origin."""
    return np.c_[(P[:, 0] + org[0]) / sp[0], (P[:, 1] + org[1]) / sp[1], (P[:, 2] - org[2]) / sp[2]]

def plaque(ct, mask, sp, org, pts, refs, thr):
    ijk = to_index(pts, sp, org)
    pad = (refs.max() + SHELL_MM + 1) / sp
    lo = np.maximum(np.floor(ijk.min(0) - pad).astype(int), 0)
    hi = np.minimum(np.ceil(ijk.max(0) + pad).astype(int), np.array(ct.shape[::-1]) - 1)
    xs, ys, zs = (np.arange(lo[a], hi[a] + 1) for a in range(3))
    Z, Y, X = np.meshgrid(zs, ys, xs, indexing="ij")
    world = np.c_[X.ravel() * sp[0] - org[0], Y.ravel() * sp[1] - org[1], Z.ravel() * sp[2] + org[2]]
    d, nn = cKDTree(pts).query(world)
    within = d <= refs[nn] + SHELL_MM
    hu = ct[lo[2]:hi[2] + 1, lo[1]:hi[1] + 1, lo[0]:hi[0] + 1].ravel()
    lum = mask[lo[2]:hi[2] + 1, lo[1]:hi[1] + 1, lo[0]:hi[0] + 1].ravel() == 1
    ca = within & (hu > thr)
    wall = within & ~lum
    return ca.sum() * sp.prod(), (float(np.median(hu[wall])) if wall.any() else np.nan)

def cpr(ct, mask, sp, org, z, p, zc, half_w):
    s = np.arange(max(0.0, zc - CPR_HALF_MM), min(z[-1], zc + CPR_HALF_MM), 0.1)
    P = np.c_[[np.interp(s, z, p[:, a]) for a in range(3)]].T
    T = np.gradient(P, axis=0); T /= np.linalg.norm(T, axis=1)[:, None]
    n = np.cross(T[0], [0, 0, 1] if abs(T[0][2]) < 0.9 else [1, 0, 0]); n /= np.linalg.norm(n)
    N = np.empty_like(T)
    for i in range(len(T)):                      # parallel transport keeps the view from twisting
        n = n - (n @ T[i]) * T[i]; n /= np.linalg.norm(n); N[i] = n
    B = np.cross(T, N)
    lat = np.arange(-half_w, half_w + 1e-9, 0.05)
    views = []
    for V in (N, B):
        Q = P[None, :, :] + lat[:, None, None] * V[None, :, :]
        ijk = to_index(Q.reshape(-1, 3), sp, org)
        coords = [ijk[:, 2], ijk[:, 1], ijk[:, 0]]
        img = map_coordinates(ct, coords, order=1, mode="nearest").reshape(len(lat), len(s))
        msk = map_coordinates(mask.astype(np.float32), coords, order=1, mode="nearest").reshape(len(lat), len(s))
        views.append((img, msk))
    return s - zc, lat, views

In [ ]:
def analyse(name, reported):
    mesh = trimesh.load(os.path.join(DISEASED, "SurfaceMeshes", f"{name}.stl"))
    mask, sp, org = read_nrrd(os.path.join(DISEASED, "Annotations", f"{name}.nrrd"))
    ct, _, _ = read_nrrd(os.path.join(DISEASED, "CTCA", f"{name}.nrrd"))
    ct = ct.astype(np.float32)
    lum = ct[mask == 1]
    thr = max(CA_MIN_HU, float(lum.mean() + 3 * lum.std()))

    paths, prof, bifs = path_profiles(name)
    lesions = []
    for b, P in enumerate(prof):
        z, r, p, rs, ref, ds = P["z"], P["r"], P["p"], P["rs"], P["ref"], P["ds"]
        for a, e in find_lesions(z, ds):
            k = a + int(np.argmax(ds[a:e + 1]))
            j = a + int(np.argmin(r[a:e + 1]))
            dbif = np.min(np.linalg.norm(bifs - p[k], axis=1)) if len(bifs) else np.inf
            # tips: branches can end where the annotators stopped (vessel < 1 mm or imaging artefact),
            # so a narrowing only counts if healthy vessel is seen again past it
            after = np.arange(e + 1, len(z))
            ok_run, recovers = 0.0, False
            for i in after:
                ok_run = ok_run + (z[i] - z[i - 1]) if ds[i] < RECOVER_DS else 0.0
                if ok_run >= RECOVER_MM: recovers = True; break
            runoff = z[-1] - z[e]
            lesions.append(dict(path=b, a=a, e=e, k=k, z_throat=z[k], z_start=z[a], z_end=z[e],
                                length=z[e] - z[a], ref_r=ref[k], throat_rs=rs[k], min_r_raw=r[j],
                                ds=ds[k], ds_raw=1 - r[j] / ref[j], xyz=p[k],
                                near_ostium=z[k] < NEAR_MM, near_tip=(z[-1] - z[k] < NEAR_MM) or not recovers,
                                runoff=runoff, short_runoff=runoff < SHORT_RUNOFF_MM,
                                near_bif=dbif < NEAR_MM + ref[k]))
    # merge duplicates from shared trunks (same throat within 2 mm): keep the most severe
    lesions.sort(key=lambda L: -L["ds"])
    kept = []
    for L in lesions:
        if all(np.linalg.norm(L["xyz"] - K["xyz"]) > 2.0 for K in kept): kept.append(L)
    lesions = kept

    # mesh and CT checks
    for L in lesions:
        P = prof[L["path"]]; z, p, rs, ref, ds = P["z"], P["p"], P["rs"], P["ref"], P["ds"]
        sel = np.where((z >= L["z_start"] - 3) & (z <= L["z_end"] + 3))[0]
        cen = p[sel]
        R = 3 * ref[sel].max() + 3
        fc = mesh.triangles_center
        near = np.where(cKDTree(cen).query(fc)[0] < R)[0]
        sub = mesh.submesh([near], append=True)
        step = max(1, int(round(0.25 / max(np.median(np.diff(z)), 1e-3))))
        ii = sel[::step]
        eq = np.array([section_eq_r(sub, p[i], tangent(p, i)) for i in ii])
        inside = (z[ii] >= L["z_start"]) & (z[ii] <= L["z_end"])
        shoulder = (~inside) & (ds[ii] < 0.15) & np.isfinite(eq)
        kfac = np.median(eq[shoulder] / rs[ii][shoulder]) if shoulder.sum() >= 3 else 1.1
        good = inside & np.isfinite(eq)
        if good.any():
            m = np.argmin(eq[good])
            L["mesh_ds"] = 1 - eq[good][m] / (kfac * ref[ii][good][m])
        else:
            L["mesh_ds"] = np.nan
        a, e = L["a"], L["e"]
        L["ca_mm3"], L["wall_hu"] = plaque(ct, mask, sp, org, p[a:e + 1], ref[a:e + 1], thr)

    for L in lesions:
        L["mesh_disagrees"] = bool(np.isnan(L["mesh_ds"]) or abs(L["ds"] - L["mesh_ds"]) > 0.2)
    # only clean lesions can be picked; if there are none the case needs a manual choice in overrides.csv
    cand = [L for L in lesions if not (L["near_tip"] or L["near_ostium"] or L["mesh_disagrees"])]
    pick = max(cand, key=lambda L: L["ds"]) if cand else None
    for i, L in enumerate(lesions):
        L["id"] = f"L{i + 1}"
        L["picked"] = L is pick
    return dict(name=name, reported=reported, paths=paths, prof=prof, lesions=lesions, pick=pick,
                mesh=mesh, ct=ct, mask=mask, sp=sp, org=org, thr=thr, bifs=bifs)

In [ ]:
FIELDS = ["case", "lesion", "picked", "path", "throat_z_mm", "start_z_mm", "end_z_mm", "length_mm",
          "reference_r_mm", "throat_r_smoothed_mm", "min_r_raw_mm", "narrowing", "narrowing_raw",
          "narrowing_mesh", "calcium_mm3", "wall_median_hu", "near_ostium", "near_tip", "near_bifurcation", "mesh_disagrees", "runoff_mm", "short_runoff",
          "reported_severity", "throat_x", "throat_y", "throat_z"]

def lesion_rows(res):
    return [[res["name"], L["id"], L["picked"], L["path"], round(L["z_throat"], 2), round(L["z_start"], 2),
             round(L["z_end"], 2), round(L["length"], 2), round(L["ref_r"], 3), round(L["throat_rs"], 3),
             round(L["min_r_raw"], 3), round(L["ds"], 3), round(L["ds_raw"], 3), round(L["mesh_ds"], 3),
             round(L["ca_mm3"], 2), round(L["wall_hu"], 0), L["near_ostium"], L["near_tip"], L["near_bif"], L["mesh_disagrees"], round(L["runoff"], 1), L["short_runoff"],
             res["reported"], *np.round(L["xyz"], 2)] for L in res["lesions"]]

def draw(res):
    name, les, pick = res["name"], res["lesions"], res["pick"]
    ok = lambda L: not (L["near_tip"] or L["near_ostium"] or L["mesh_disagrees"])
    ranked = [L for L in les if L is not pick and ok(L)] or [L for L in les if L is not pick]
    runner = ranked[0] if ranked else None
    show = [x for x in (pick, runner) if x is not None]
    if not show:
        fig = plt.figure(figsize=(8, 2)); fig.text(0.5, 0.5, f"{name}: no narrowings over {LESION_DS:.0%}", ha="center")
        out = os.path.join(HERE, f"{name}_review.png"); fig.savefig(out); plt.close(fig); return out
    prof_rows = [show[0]] + [L for L in show[1:] if L["path"] != show[0]["path"]]
    nr = len(prof_rows)
    fig = plt.figure(figsize=(17, 11 + 3.2 * nr))
    gs = fig.add_gridspec(nr + 2, 2, height_ratios=[1.1] * nr + [1.3, 1.4], hspace=0.45, wspace=0.18, top=0.95)
    fig.suptitle(f"{name}  |  reported most severe obstruction {res['reported']:.0%}  |  "
                 + (f"picked {pick['id']} ({pick['ds']:.0%} narrowing)" if pick else
                    "NO CLEAN LESION — every narrowing is flagged; choose one in overrides.csv"),
                 fontsize=15, fontweight="bold", y=0.985)

    for row, L in enumerate(prof_rows):
        ax = fig.add_subplot(gs[row, :])
        P = res["prof"][L["path"]]
        ax.plot(P["z"], P["r"], color="0.75", lw=0.8, label="radius (raw)")
        ax.plot(P["z"], P["rs"], color="k", lw=1.4, label="radius (1 mm average)")
        ax.plot(P["z"], P["ref"], "--", color="tab:blue", lw=1.4, label="healthy reference (taper fit)")
        for M in les:
            if M["path"] == L["path"] or np.min(np.linalg.norm(P["p"] - M["xyz"], axis=1)) < 0.3:
                kk = int(np.argmin(np.linalg.norm(P["p"] - M["xyz"], axis=1)))
                span = (P["z"] >= P["z"][kk] - (M["z_throat"] - M["z_start"])) & (P["z"] <= P["z"][kk] + (M["z_end"] - M["z_throat"]))
                col = "tab:red" if M is pick else ("tab:orange" if M is runner else "tab:gray")
                ax.fill_between(P["z"], 0, P["ref"].max() * 1.15, where=span, color=col, alpha=0.18)
                ax.annotate(f"{M['id']} {M['ds']:.0%}", (P["z"][kk], P["rs"][kk]), xytext=(0, -26),
                            textcoords="offset points", ha="center", color=col, fontweight="bold")
        for bp in res["bifs"]:
            d = np.linalg.norm(P["p"] - bp, axis=1)
            if d.min() < 0.3: ax.axvline(P["z"][np.argmin(d)], color="tab:green", ls=":", lw=1)
        ax.set_xlim(0, P["z"][-1]); ax.set_ylim(0, P["ref"].max() * 1.15)
        ax.set_xlabel("distance along centreline from ostium (mm)"); ax.set_ylabel("radius (mm)")
        ax.set_title(f"Path {L['path']} radius profile — " + ", ".join(f"{'picked' if M is pick else 'runner-up'} {M['id']}" for M in show if M['path'] == L['path']) + " "
                     f"(green dotted = bifurcation)", loc="left")
        if row == 0: ax.legend(loc="upper right", fontsize=8, ncol=3)

    for col, L in enumerate(show):
        P = res["prof"][L["path"]]
        half_w = max(2.5, 2.2 * L["ref_r"])
        s, lat, views = cpr(res["ct"], res["mask"], res["sp"], res["org"], P["z"], P["p"], L["z_throat"], half_w)
        sub = gs[nr, col].subgridspec(2, 1, hspace=0.12)
        for v, (img, msk) in enumerate(views):
            ax = fig.add_subplot(sub[v])
            ax.imshow(img, cmap="gray", vmin=-100, vmax=700, aspect="auto", origin="lower",
                      extent=[s[0], s[-1], lat[0], lat[-1]])
            ax.contour(s, lat, msk, levels=[0.5], colors="cyan", linewidths=0.8)
            ca = np.ma.masked_less(img, res["thr"])
            ax.imshow(np.ones_like(ca), cmap="autumn", alpha=np.where(ca.mask, 0, 0.6), aspect="auto", origin="lower",
                      extent=[s[0], s[-1], lat[0], lat[-1]])
            ax.axvline(L["z_start"] - L["z_throat"], color="yellow", lw=0.8, ls="--")
            ax.axvline(L["z_end"] - L["z_throat"], color="yellow", lw=0.8, ls="--")
            if v == 0:
                ax.set_title(f"{L['id']} straightened CT ({'picked' if L is pick else 'runner-up'}), two views at 90°\n"
                             f"cyan = lumen mask, orange = calcium (> {res['thr']:.0f} HU), yellow = lesion ends",
                             fontsize=9, loc="left")
                ax.set_xticklabels([])
            else:
                ax.set_xlabel("mm from throat (flow →)")
            ax.set_ylabel("mm")

    ax = fig.add_subplot(gs[nr + 1, 0], projection="3d")
    v = res["mesh"].vertices[:: max(1, len(res["mesh"].vertices) // 6000)]
    ax.scatter(v[:, 0], v[:, 1], v[:, 2], s=0.3, c="0.8", alpha=0.3)
    for P in res["prof"]:
        ax.plot(P["p"][:, 0], P["p"][:, 1], P["p"][:, 2], color="0.5", lw=0.6)
    for M in les:
        col = "tab:red" if M is pick else ("tab:orange" if M is runner else "tab:blue")
        ax.scatter(*M["xyz"], s=60 if M in show else 25, c=col, marker="*" if M is pick else "o")
        ax.text(*M["xyz"], f" {M['id']}", fontsize=8, color=col)
    ax.set_title("Where the lesions are (red star = picked, orange = runner-up)", fontsize=10)
    ax.set_axis_off()

    ax = fig.add_subplot(gs[nr + 1, 1]); ax.axis("off")
    hdr = ["id", "path", "z mm", "len", "narrow", "mesh", "Ca mm³", "flags"]
    rows = []
    for M in les[:14]:
        fl = ",".join(f for f, c in (("ostium", M["near_ostium"]), ("tip", M["near_tip"]), ("bif", M["near_bif"]), ("mesh≠", M["mesh_disagrees"]), (f"runoff {M['runoff']:.0f}mm", M["short_runoff"] and not M["near_tip"])) if c)
        rows.append([M["id"] + (" ◀" if M is pick else ""), M["path"], f"{M['z_throat']:.1f}", f"{M['length']:.1f}",
                     f"{M['ds']:.0%}", "–" if np.isnan(M["mesh_ds"]) else f"{M['mesh_ds']:.0%}", f"{M['ca_mm3']:.1f}", fl])
    t = ax.table(cellText=rows, colLabels=hdr, loc="upper center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8.5); t.scale(1, 1.25)
    ax.set_title("All lesions (narrowing > 25%), most severe first", fontsize=10)
    out = os.path.join(HERE, f"{name}_review.png")
    import shutil, tempfile, time
    tmp = os.path.join(tempfile.gettempdir(), os.path.basename(out))
    fig.savefig(tmp, dpi=110, bbox_inches="tight"); plt.close(fig)
    for attempt in range(10):          # synced folders (OneDrive) can briefly lock a file
        try:
            shutil.copyfile(tmp, out); break
        except OSError:
            time.sleep(1)
    return out

def rebuild_tables():
    rows = []
    for f in sorted(glob.glob(os.path.join(HERE, "Diseased_*_lesions.csv")), key=lambda f: int(os.path.basename(f).split("_")[1])):
        rows += list(csv.reader(open(f)))[1:]
    with open(os.path.join(HERE, "lesions_all.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(FIELDS); w.writerows(rows)
    with open(os.path.join(HERE, "picks.csv"), "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["case", "reported_severity", "picked_lesion", "path", "throat_z_mm", "narrowing", "narrowing_mesh",
                    "calcium_mm3", "runoff_mm", "short_runoff", "n_lesions", "difference_vs_reported"])
        cases = {}
        for r in rows: cases.setdefault(r[0], []).append(dict(zip(FIELDS, r)))
        for c, rr in cases.items():
            p = [r for r in rr if r["picked"] == "True"]
            if not p:
                w.writerow([c, rr[0]["reported_severity"], "none (no clean lesion)", "", "", "", "", "", "", "", len(rr), ""])
            if p:
                p = p[0]
                w.writerow([c, p["reported_severity"], p["lesion"], p["path"], p["throat_z_mm"], p["narrowing"],
                            p["narrowing_mesh"], p["calcium_mm3"], p["runoff_mm"], p["short_runoff"], len(rr),
                            round(float(p["narrowing"]) - float(p["reported_severity"]), 3)])
    # flag_definitions.csv: what each flag in lesions_all.csv means (thresholds follow the settings above)
    with open(os.path.join(HERE, "flag_definitions.csv"), "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.writer(fh)
        w.writerow(["flag_on_review_page", "csv_column", "meaning", "why_it_matters", "can_be_picked"])
        w.writerows([
            ["ostium", "near_ostium",
             f"The narrowest point is within {NEAR_MM:g} mm of the start of the artery, where it leaves the aorta.",
             "The centreline starts inside the aortic wall, so the radius there is often artificially small; a dip that close is usually a measurement effect, not disease.",
             "No"],
            ["tip", "near_tip",
             f"The narrowest point is within {NEAR_MM:g} mm of the end of the branch, or the artery never recovers after the narrowing "
             f"(never back to less than {RECOVER_DS:.0%} narrowed for at least {RECOVER_MM:g} mm before the branch ends).",
             "Branches end where the annotators stopped (vessel under 1 mm across, or the image too poor), so a narrowing that runs into the end may just be the vessel fading out.",
             "No"],
            ["bif", "near_bifurcation",
             f"The narrowest point is within about {NEAR_MM:g} mm plus one vessel radius of a branch point.",
             "Where a side branch splits off, the lumen changes shape and the radius dips, so a real narrowing is harder to tell apart from branch-point geometry.",
             "Yes, but treat with care"],
            ["mesh≠", "mesh_disagrees",
             "The narrowing measured from the centreline radius and from the mesh cross-section differ by more than 20 percentage points.",
             "The two should roughly agree; a big gap usually means one of them is fooled, e.g. by an oval or pinched cross-section.",
             "No"],
            ["runoff Xmm", "short_runoff (length in runoff_mm)",
             f"The artery recovers after the narrowing, but there is less than {SHORT_RUNOFF_MM:g} mm of it before the branch ends. X is that length in mm.",
             "Probably a real lesion, but a clip cannot get its full 7D downstream and the short stretch after it may be unreliable.",
             "Yes, flagged for checking"],
        ])

    # overrides.csv: your sign-off. Existing rows are never touched; new cases are appended.
    ov = os.path.join(HERE, "overrides.csv")
    have = [r[0] for r in csv.reader(open(ov))][1:] if os.path.exists(ov) else None
    with open(ov, "a" if have is not None else "w", newline="") as fh:
        w = csv.writer(fh)
        if have is None:
            w.writerow(["case", "use_lesion", "or_path", "or_throat_z_mm", "notes"]); have = []
        for c in cases:
            if c not in have: w.writerow([c, "", "", "", ""])

In [ ]:
def run_case(name):
    """Analyse one case, write its lesion CSV and review page, and return the result."""
    res = analyse(name, reported[name])
    with open(os.path.join(HERE, f"{name}_lesions.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(FIELDS); w.writerows(lesion_rows(res))
    png = draw(res)
    pk = res["pick"]
    print(f"{name}: {len(res['lesions'])} lesions; picked {pk['id']} path {pk['path']} z={pk['z_throat']:.1f} "
          f"narrowing {pk['ds']:.0%} (mesh {pk['mesh_ds']:.0%}, Ca {pk['ca_mm3']:.1f} mm3) vs reported {res['reported']:.0%}"
          if pk else f"{name}: {len(res['lesions'])} lesions, none clean (all at the ostium/tip or disagreeing with the mesh) "
                     f"— no pick; choose one in overrides.csv if you want this case cut", flush=True)
    return res, png

ws = openpyxl.load_workbook(os.path.join(DISEASED, "DiseaseReports.xlsx"), data_only=True).worksheets[0]
reported = {row[0]: row[2] for row in ws.iter_rows(min_row=2, values_only=True) if row[0]}
CASES = sorted([n for n, v in reported.items() if v >= MIN_REPORTED], key=lambda n: int(n.split("_")[1]))
print("Cases reported >=", MIN_REPORTED, ":", CASES)

## 6. Run a single case
Change `CASE` and run this cell. The review page is shown below and saved to the folder.

In [ ]:
from IPython.display import Image, display
CASE = "Diseased_1"
res, png = run_case(CASE)
display(Image(filename=png))

## 7. Run all cases
Runs every case in `CASES` (about 15–25 s each). Edit `CASES` in section 5 or here to run a subset, e.g. `CASES = ['Diseased_6', 'Diseased_12']`.

In [ ]:
results = {}
for name in CASES:
    results[name], _ = run_case(name)

## 8. Rebuild the combined tables
Combines every `Diseased_N_lesions.csv` in the folder into `lesions_all.csv`, `picks.csv` and `flag_definitions.csv`, and adds any new cases to `overrides.csv` (existing rows are never changed).

In [ ]:
rebuild_tables()
print('Tables rebuilt in', HERE)

## 9. Look at the tables (optional)

In [ ]:
import pandas as pd
picks = pd.read_csv(os.path.join(HERE, "picks.csv"))
display(picks)
lesions = pd.read_csv(os.path.join(HERE, "lesions_all.csv"))
display(lesions[lesions["case"] == CASE])
display(pd.read_csv(os.path.join(HERE, "flag_definitions.csv")))

## 10. Cut the clips
Uses each case's picked lesion, or your choice from `overrides.csv` (`use_lesion`, or `or_path` + `or_throat_z_mm`).
Needs sections 1–5; it does not read the CT, and it only needs the `Diseased_N_lesions.csv` files from section 6/7.

For each case:
- **Start:** walk upstream from the lesion until the vessel has been healthy (< 15% narrowed against the reference) for `SUSTAIN_MM`.
- **End:** walk downstream from the lesion the same way, but never stop before `DOWN_D` × D past the throat (D = throat diameter).
  If the vessel never stays healthy that long before the branch ends, the clip stops at the healthiest point
  after the lesion (ignoring the last 3 mm, where annotations fade out) and is flagged.
- **Tandem lesions:** another narrowing (> 25%) within `TANDEM_MM` of either end is taken into the clip and the search repeats
  (narrowings in the first/last 3 mm of a branch don't count).
- Writes the raw `z_mm, r_opened_mm` profile (z re-zeroed at the start), `_summary.csv`, and the STLs: the mesh is cut only across
  the target vessel at both ends, giving `<case>_clip.stl` and `<case>_rest.stl` (both capped, CT coordinates, nothing deleted).

The STL step also needs `mapbox_earcut` (`pip install mapbox_earcut`).

In [ ]:
CLIP_DIR = os.path.join(DISEASED, "StenosisClips_Reviewed")
STL_DIR = CLIP_DIR + "_STL"
DOWN_D = 7            # minimum clip length downstream of the throat, in throat diameters
SUSTAIN_MM = 2.0      # vessel must stay healthy (< RECOVER_DS narrowed) this long to count as recovered
TANDEM_MM = 5.0       # another narrowing this close to a clip end is taken into the clip
MAKE_STL = True

In [ ]:
def chosen_lesion(name):
    """The lesion to cut around: your override if there is one, otherwise the automatic pick."""
    rows = list(csv.DictReader(open(os.path.join(HERE, f"{name}_lesions.csv"))))
    ovs = {r["case"]: r for r in csv.DictReader(open(os.path.join(HERE, "overrides.csv")))}
    ov = {k: (v or "").strip() for k, v in ovs.get(name, {}).items()}
    if ov.get("use_lesion"):
        L = next(r for r in rows if r["lesion"] == ov["use_lesion"])
        return dict(path=int(L["path"]), z=float(L["throat_z_mm"]), lesion=L["lesion"], source="override (lesion)")
    if ov.get("or_path") and ov.get("or_throat_z_mm"):
        return dict(path=int(ov["or_path"]), z=float(ov["or_throat_z_mm"]), lesion="manual", source="override (path, z)")
    L = next((r for r in rows if r["picked"] == "True"), None)
    if L is None:
        return None
    return dict(path=int(L["path"]), z=float(L["throat_z_mm"]), lesion=L["lesion"], source="automatic pick")

def recovered_at(z, ds, i, step):
    """Walk from station i (step -1 = upstream, +1 = downstream) to the first station where the vessel
    has been healthy for SUSTAIN_MM. None if the branch ends first."""
    run, j = 0.0, i
    while 0 <= j + step < len(z):
        j += step
        run = run + abs(z[j] - z[j - step]) if ds[j] < RECOVER_DS else 0.0
        if run >= SUSTAIN_MM: return j
    return None

def narrowed_stretch(ds, k):
    """First and last station of the narrowed stretch (> LESION_DS) around station k."""
    a = e = k
    while a > 0 and ds[a - 1] > LESION_DS: a -= 1
    while e < len(ds) - 1 and ds[e + 1] > LESION_DS: e += 1
    return a, e

def clip_limits(P, k):
    """Clip start / end station around throat k, following the rules above."""
    z, r, ds = P["z"], P["r"], P["ds"]
    D = 2 * r[k]
    a, e = narrowed_stretch(ds, k)
    tandem = []
    while True:
        i0 = recovered_at(z, ds, a, -1)
        i0 = 0 if i0 is None else i0
        i1 = recovered_at(z, ds, e, +1)
        recovered_down = i1 is not None
        i7 = min(int(np.searchsorted(z, z[k] + DOWN_D * D, "left")), len(z) - 1)   # first station at or past 7D
        if not recovered_down:
            # never healthy for SUSTAIN_MM before the tip: stop at its healthiest point (outside the tip zone)
            after = np.where((np.arange(len(z)) > e) & (z <= z[-1] - NEAR_MM))[0]
            i1 = int(after[np.argmin(ds[after])]) if len(after) else len(z) - 1
        i1 = min(max(i1, i7), len(z) - 1)
        # narrowings in the ostium / tip zones are measurement or annotation effects, not tandem lesions
        idx = np.arange(len(z))
        body = (z >= NEAR_MM) & (z <= z[-1] - NEAR_MM)
        up = np.where((z >= z[i0] - TANDEM_MM) & (idx < i0) & (ds > LESION_DS) & body)[0]
        dn = np.where((z <= z[i1] + TANDEM_MM) & (idx > i1) & (ds > LESION_DS) & body)[0]
        if not len(up) and not len(dn): break
        if len(up): a = narrowed_stretch(ds, up[-1])[0]; tandem.append(round(float(z[up[-1]]), 1))
        if len(dn): e = narrowed_stretch(ds, dn[0])[1]; tandem.append(round(float(z[dn[0]]), 1))
    return dict(i0=i0, i1=i1, D=D, recovered_down=recovered_down, full_7D=z[i1] - z[k] >= DOWN_D * D - 1e-6,
                reached_ostium=i0 == 0, reached_tip=i1 == len(z) - 1, tandem=sorted(set(tandem)))

In [ ]:
def local_cut(mesh, origin, normal, target):
    """Cut the mesh along a plane, but only through the vessel cross-section containing `target`.
    Both sides are capped; other vessels the plane happens to cross are left intact."""
    V, F = mesh.vertices, mesh.faces
    d = (V - origin) @ normal
    d[np.abs(d) < 1e-9] = 1e-9
    side = d > 0
    crossing = np.where(side[F].any(1) & ~side[F].all(1))[0]
    # group crossing faces into rings and keep the ring that encircles the target point
    adj = mesh.face_adjacency
    keep = np.isin(adj, crossing).all(1)
    rings = trimesh.graph.connected_components(adj[keep], nodes=crossing)
    u = np.cross(normal, [1, 0, 0] if abs(normal[0]) < 0.9 else [0, 1, 0]); u /= np.linalg.norm(u)
    v = np.cross(normal, u)
    to2d = lambda P: np.c_[(P - origin) @ u, (P - origin) @ v]
    edge_pt = lambda a, b: V[a] + (V[b] - V[a]) * (d[a] / (d[a] - d[b]))[:, None]

    def ring_loop(faces):
        segs = []
        for f in F[faces]:
            k = [i for i in range(3) if side[f[i]] != side[f[(i + 1) % 3]] or side[f[i]] != side[f[(i + 2) % 3]]]
            lone = [i for i in range(3) if side[f[i]] != side[f[(i + 1) % 3]] and side[f[i]] != side[f[(i + 2) % 3]]][0]
            a, b, c = f[lone], f[(lone + 1) % 3], f[(lone + 2) % 3]
            segs.append((tuple(sorted((a, b))), tuple(sorted((a, c)))))
        nbr = {}
        for e1, e2 in segs:
            nbr.setdefault(e1, []).append(e2); nbr.setdefault(e2, []).append(e1)
        loop, prev = [segs[0][0]], None
        while True:
            nxt = [e for e in nbr[loop[-1]] if e != prev][0]
            if nxt == loop[0]: break
            prev = loop[-1]; loop.append(nxt)
        return loop

    from shapely.geometry import Point, Polygon
    tp = Point(to2d(target[None])[0])
    ring = None
    for rf in rings:
        rf = np.asarray(rf)
        if np.min(np.linalg.norm(mesh.triangles_center[rf] - target, axis=1)) > 20: continue
        loop = ring_loop(rf)
        pts = edge_pt(np.array([e[0] for e in loop]), np.array([e[1] for e in loop]))
        if Polygon(to2d(pts)).contains(tp):
            ring, ring_edges, ring_pts = rf, loop, pts
            break
    if ring is None:
        raise RuntimeError("no vessel cross-section around the centreline point")

    # new vertices on the cut: one copy for each side so the two halves separate
    n0 = len(V)
    key = {e: i for i, e in enumerate(ring_edges)}
    plus = lambda e: n0 + key[e]
    minus = lambda e: n0 + len(ring_edges) + key[e]
    newF = []
    for f in F[ring]:
        lone = [i for i in range(3) if side[f[i]] != side[f[(i + 1) % 3]] and side[f[i]] != side[f[(i + 2) % 3]]][0]
        a, b, c = f[lone], f[(lone + 1) % 3], f[(lone + 2) % 3]
        eab, eac = tuple(sorted((a, b))), tuple(sorted((a, c)))
        A, O = (plus, minus) if side[a] else (minus, plus)      # copy for a's side / the other side
        newF += [(a, A(eab), A(eac)), (O(eab), b, c), (O(eab), c, O(eac))]
    # caps: triangulate the cross-section once, use it for both sides with opposite winding
    import mapbox_earcut
    tri = mapbox_earcut.triangulate_float64(to2d(ring_pts), np.array([len(ring_pts)], dtype=np.uint32)).reshape(-1, 3)
    n1 = np.cross(ring_pts[tri[0, 1]] - ring_pts[tri[0, 0]], ring_pts[tri[0, 2]] - ring_pts[tri[0, 0]])
    if n1 @ normal > 0: tri = tri[:, ::-1]                       # plus-side cap must face -normal
    idx = np.arange(len(ring_edges))
    cap_plus = n0 + idx[tri]
    cap_minus = n0 + len(ring_edges) + idx[tri][:, ::-1]
    faces = np.vstack([np.delete(F, ring, axis=0), np.array(newF), cap_plus, cap_minus])
    verts = np.vstack([V, ring_pts, ring_pts])
    return trimesh.Trimesh(verts, faces, process=False)

def cut_stl(name, p, i0, i1, k):
    """Cut the case's mesh across the target vessel at stations i0 and i1; write <case>_clip.stl and <case>_rest.stl."""
    m = trimesh.load(os.path.join(DISEASED, "SurfaceMeshes", f"{name}.stl"))
    m = local_cut(m, p[i0], tangent(p, i0), p[i0])
    m = local_cut(m, p[i1], tangent(p, i1), p[i1])
    parts = m.split(only_watertight=False)
    # the clip is the piece that holds the centreline between the two cuts (the throat itself can sit
    # right against the wall, so test several stations rather than just the throat)
    inner = p[i0 + 1:i1][:: max(1, (i1 - i0) // 30)]
    j = int(np.argmin([np.median(cKDTree(b.vertices).query(inner)[0]) for b in parts]))
    rest = trimesh.util.concatenate([b for i, b in enumerate(parts) if i != j])
    parts[j].export(os.path.join(STL_DIR, f"{name}_clip.stl"))
    rest.export(os.path.join(STL_DIR, f"{name}_rest.stl"))
    return parts[j].is_watertight and all(b.is_watertight for i, b in enumerate(parts) if i != j)

In [ ]:
CUT_FIELDS = ["case", "lesion", "source", "path", "throat_z_on_path_mm", "D_mm", "healthy_radius_mm", "throat_radius_mm",
              "narrowing", "upstream_mm", "downstream_mm", "downstream_7D_mm", "clip_length_mm", "n_stations",
              "throat_position_pct", "min_radius_position_pct", "start_narrowing", "end_narrowing",
              "full_7D_downstream", "recovered_downstream", "reached_ostium", "reached_tip", "tandem_lesions_at_z",
              "stl_watertight"]

def cut_case(name):
    ch = chosen_lesion(name)
    if ch is None:
        print(f"{name}: skipped — no clean lesion was picked; set use_lesion (or or_path + or_throat_z_mm) in overrides.csv")
        return None
    paths, prof, bifs = path_profiles(name)
    P = prof[ch["path"]]
    z, r, p, ref, ds = P["z"], P["r"], P["p"], P["ref"], P["ds"]
    k = int(np.argmin(np.abs(z - ch["z"])))
    c = clip_limits(P, k)
    i0, i1 = c["i0"], c["i1"]
    zc, rc = z[i0:i1 + 1] - z[i0], r[i0:i1 + 1]
    os.makedirs(CLIP_DIR, exist_ok=True); os.makedirs(STL_DIR, exist_ok=True)
    with open(os.path.join(CLIP_DIR, f"{name}_clip.csv"), "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["z_mm", "r_opened_mm"])
        w.writerows([(f"{a:.6f}", f"{b:.6f}") for a, b in zip(zc, rc)])
    wt = cut_stl(name, p, i0, i1, k) if MAKE_STL else ""
    L = zc[-1]
    row = [name, ch["lesion"], ch["source"], ch["path"], round(z[k], 2), round(c["D"], 3), round(ref[k], 3), round(r[k], 3),
           round(ds[k], 3), round(z[k] - z[i0], 2), round(z[i1] - z[k], 2), round(DOWN_D * c["D"], 2), round(L, 2), len(zc),
           round(100 * (z[k] - z[i0]) / L, 1), round(100 * zc[int(np.argmin(rc))] / L, 1), round(ds[i0], 2), round(ds[i1], 2),
           c["full_7D"], c["recovered_down"], c["reached_ostium"], c["reached_tip"], " ".join(map(str, c["tandem"])), wt]
    flags = [f for f, v in (("7D not reached", not c["full_7D"]), ("no downstream recovery", not c["recovered_down"]),
                            ("hit ostium", c["reached_ostium"]), ("hit tip", c["reached_tip"])) if v]
    print(f"{name}: {ch['lesion']} ({ch['source']}) path {ch['path']} z={z[k]:.1f}  up {z[k]-z[i0]:.1f} / down {z[i1]-z[k]:.1f} mm "
          f"(7D={DOWN_D*c['D']:.1f})  clip {L:.1f} mm, {len(zc)} stations"
          + (f"  tandem at z={c['tandem']}" if c["tandem"] else "") + (f"  ⚠ {', '.join(flags)}" if flags else ""), flush=True)
    return row

def write_cut_summary(rows):
    path = os.path.join(CLIP_DIR, "_summary.csv")
    old = {r[0]: r for r in list(csv.reader(open(path)))[1:]} if os.path.exists(path) else {}
    old.update({r[0]: r for r in rows if r})
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(CUT_FIELDS)
        w.writerows([old[k] for k in sorted(old, key=lambda n: int(n.split("_")[1]))])

### Cut one case, or all of them
Set `CASE` to cut a single case, or run the loop for every case in `CASES`. The summary keeps rows from earlier runs.

In [ ]:
CASE = "Diseased_1"
write_cut_summary([cut_case(CASE)])

In [ ]:
write_cut_summary([cut_case(name) for name in CASES])